In [1]:
from bertopic import BERTopic
import json
import time

from nltk.tokenize import word_tokenize
from gensim.models import CoherenceModel
from gensim.corpora import Dictionary
from itertools import combinations

In [2]:
K_RANGE = list(range(3, 20))
TOP_N = 10

In [3]:
try:
    with open('../dataProcessed/nurseNotesProcessed.json', 'r') as file:
        nurse_notes = json.load(file)
    print("File loaded successfully.")
    
except FileNotFoundError:
    print("Error: The file 'data.json' was not found.")

File loaded successfully.


In [4]:
# Make utility function to get coherence. - gensim implementation is also used in OCTIS (https://github.com/MIND-Lab/OCTIS/blob/master/octis/evaluation_metrics/coherence_metrics.py)
def get_coherence_score(topics, tokenized_texts, dictionary, coherence_type):
    coherence_model = CoherenceModel(
        topics=topics,
        texts=tokenized_texts,
        dictionary=dictionary,
        coherence=coherence_type # either 'c_npmi' or 'c_v'
    )
    return coherence_model.get_coherence()

# Make utility function to get diversity (always 10 words) - adapted from OCTIS (https://github.com/MIND-Lab/OCTIS/blob/master/octis/evaluation_metrics/diversity_metrics.py)
def get_diversity_score(topics):
    if len(topics) <= 0:
        return 0.0
    unique_words = set()
    for topic in topics:
        unique_words.update(topic[:10])
    return len(unique_words) / (10 * len(topics))

# Redundancy calculation. Higher the better, as closer to 1 means non-overlapping topics (inverted score). Adapted from https://aclanthology.org/2024.acl-long.11/.
def compute_topic_redundancy(topic_word_distributions, top_n=10):
    redundancy_scores = []
    for topic1, topic2 in combinations(topic_word_distributions, 2):
        overlap = len(set(topic1[:top_n]).intersection(set(topic2[:top_n])))
        redundancy = overlap / top_n
        redundancy_scores.append(redundancy)
    average_redundancy = sum(redundancy_scores) / len(redundancy_scores)
    return 1 - average_redundancy

In [5]:
def bertopic_analysis(texts):
    tokenized_texts = [word_tokenize(text.lower()) for text in texts]
    dictionary = Dictionary(tokenized_texts)

    print(f"Number of texts: {len(texts)}")

    start = time.time() 
    topic_model = BERTopic(
        embedding_model='sentence-transformers/all-MiniLM-L6-v2',
    )
    _topics, _probs = topic_model.fit_transform(texts)
    cluster_topics = list(topic_model.get_topic_info()['Representation'])
    cluster_topics = [topic[:TOP_N] for topic in cluster_topics]
    end = time.time()

    coherence = get_coherence_score(cluster_topics, tokenized_texts, dictionary, 'c_v')
    diversity = get_diversity_score(cluster_topics)
    redundancy = compute_topic_redundancy(cluster_topics)

    print(f"Coherence: {coherence}")
    print(f"Diversity: {diversity}")
    print(f"Inverse Redundancy: {redundancy}")
    print(f"Time (seconds): {end-start}")

    print("----- Cluster Topics -----")
    for cluster_topic in cluster_topics:
        print(cluster_topic)
        
    print(f"Number of Topics: {len(cluster_topics)}")

In [6]:
all_texts = []
for key in nurse_notes.keys():
    print(f"-----------{key}-----------")
    bertopic_analysis(nurse_notes[key]['Note'])
    all_texts.extend(nurse_notes[key]['Note'])

-----------P1-----------
Number of texts: 600


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Coherence: 0.7120902162413407
Diversity: 0.5538461538461539
Inverse Redundancy: 0.8730769230769231
Time (seconds): 6.679489850997925
----- Cluster Topics -----
['weekly', 'attend', 'injection', 'restaurant', 'need', 'resident', 'give', 'med', 'good', 'administer']
['check', 'sleep', 'safety', 'night', 'settle', 'comfortable', 'continue', 'resident', 'concern', 'bed']
['walker', 'mobilizing', 'relaxed', 'take', 'staff', 'chart', 'med', 'appear', 'content', 'mg']
['eye', 'care', 'morning', 'assist', 'usual', 'give', 'form', 'chart', 'drop', 'choice']
['plan', 'morning', 'adls', 'breakfast', 'staff', 'report', 'room', 'intake', 'care', 'interact']
['restaurant', 'mobile', 'attend', 'independent', 'usual', 'need', 'take', 'meal', 'form', 'med']
['rollator', 'give', 'mobilise', 'appear', 'good', 'need', 'form', 'room', 'complaint', 'nil']
['toilette', 'ongoing', 'asleep', 'comfortable', 'self', 'check', 'resident', 'toilete', 'peacefully', 'require']
['peaceful', 'toilette', 'ongoing', 'asl

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

...

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Coherence: 0.7166839328049939
Diversity: 0.6521739130434783
Inverse Redundancy: 0.9513833992094861
Time (seconds): 6.556804180145264
----- Cluster Topics -----
['care', 'plan', 'resident', 'continue', 'check', 'night', 'safety', 'give', 'pressure', 'chart']
['settle', 'tts', 'comfortably', 'hoist', 'night', 'staff', 'medication', 'give', 'bed', 'drink']
['adls', 'adequate', 'new', 'intake', 'chart', 'nil', 'appear', 'pu', 'concern', 'med']
['vomit', 'vomiting', 'nausea', 'episode', 'feel', 'report', 'urine', 'bell', 'later', 'watch']
['wheelchair', 'baseline', 'electric', 'prescribe', 'transfer', 'wash', 'skin', 'take', 'form', 'appear']
['laxative', 'bno', 'chatty', 'natural', 'breakfast', 'content', 'morning', 'oral', 'plan', 'give']
['tele', 'early', 'watch', 'nocte', 'present', 'later', 'overnight', 'attend', 'administer', 'safety']
['protein', 'weight', 'diet', 'nutritional', 'chart', 'high', 'sinemet', 'dietitian', 'dietetic', 'wound']
['alert', 'wheel', 'chair', 'bright', 'intak

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Coherence: 0.7182188559507756
Diversity: 0.50625
Inverse Redundancy: 0.8825000000000001
Time (seconds): 6.7222740650177
----- Cluster Topics -----
['chair', 'foot', 'vaccine', 'batch', 'bed', 'resident', 'care', 'administer', 'hse', 'comirnaty']
['care', 'form', 'good', 'resident', 'attend', 'continue', 'chart', 'assist', 'nil', 'room']
['charted', 'adls', 'compliant', 'assisted', 'meds', 'have', 'maintain', 'night', 'settle', 'safety']
['adls', 'charted', 'compliant', 'assisted', 'meds', 'maintain', 'settle', 'night', 'safety', 'need']
['medication', 'administer', 'alert', 'bright', 'appear', 'new', 'house', 'eating', 'drink', 'concern']
['asleep', 'mat', 'place', 'sensor', 'comfortable', 'check', 'appear', 'chart', 'med', 'issue']
['toilet', 'nocte', 'comfortable', 'early', 'bed', 'asleep', 'check', 'place', 'sensor', 'mat']
['toilete', 'gong', 'place', 'mat', 'sensor', 'assist', 'settle', 'take', 'med', 'safety']
['care', 'skin', 'check', 'med', 'asleep', 'comfortable', 'concern', '

In [7]:
bertopic_analysis(all_texts)

Number of texts: 12191


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Coherence: 0.5903112147512469
Diversity: 0.3059880239520958
Inverse Redundancy: 0.9668932405459352
Time (seconds): 115.72430324554443
----- Cluster Topics -----
['gp', 'pain', 'doctor', 'note', 'fair', 'prn', 'lunch', 'take', 'chart', 'morning']
['tts', 'tv', 'clothe', 'midnight', 'hoist', 'pad', 'female', 'drink', 'usher', 'staff']
['change', 'report', 'plan', 'hourly', 'observation', 'sleep', 'pleasant', 'pleasantly', 'continue', 'nocte']
['complaint', 'voice', 'potter', 'medication', 'slt', 'attend', 'nil', 'rolator', 'activity', 'take']
['till', 'receive', 'ensure', 'keep', 'time', 'note', 'meet', 'observe', 'room', 'neutral']
['inhaler', 'club', 'social', 'nebs', 'aspiration', 'antibiotic', 'laxose', 'listen', 'music', 'knitting']
['mobility', 'baseline', 'wash', 'prescribe', 'dining', 'unit', 'durogesic', 'rollator', 'immediate', 'meal']
['alert', 'medicine', 'food', 'bright', 'fluid', 'render', 'intake', 'today', 'intact', 'pu']
['mobile', 'restaurant', 'independent', 'usual', '